# Factory Waste Reduction | Bronze Data Validation

## Purpose
This notebook verifies that the raw CSV files loaded into Databricks are complete and usable before any cleaning occurs. Bronze tables remain unchanged; quality issues are documented for the Silver layer.

**Source tables:** daily consumption, waste log, purchase orders, and product master (2025 and 2026 where applicable).

## 1. Confirm raw tables loaded

In [ ]:
%sql
SELECT 'bronze_daily_consumption_2025' AS table_name, COUNT(*) AS row_count FROM bronze_daily_consumption_2025
UNION ALL SELECT 'bronze_daily_consumption_2026', COUNT(*) FROM bronze_daily_consumption_2026
UNION ALL SELECT 'bronze_waste_log_2025', COUNT(*) FROM bronze_waste_log_2025
UNION ALL SELECT 'bronze_waste_log_2026', COUNT(*) FROM bronze_waste_log_2026
UNION ALL SELECT 'bronze_purchase_orders_2025', COUNT(*) FROM bronze_purchase_orders_2025
UNION ALL SELECT 'bronze_purchase_orders_2026', COUNT(*) FROM bronze_purchase_orders_2026
UNION ALL SELECT 'product_master', COUNT(*) FROM manufacturing_cost_reduction.product_master
ORDER BY table_name;

## 2. Profile raw waste data

This highlights the issues the Silver notebook will standardize, such as inconsistent waste-reason capitalization or unexpected values.

In [ ]:
%sql
SELECT
  waste_reason,
  COUNT(*) AS waste_events,
  SUM(units_wasted) AS units_wasted
FROM bronze_waste_log_2026
GROUP BY waste_reason
ORDER BY units_wasted DESC, waste_reason;

## 3. Raw-data quality checks

A good Bronze notebook reports issues; it does not fix them.

In [ ]:
%sql
SELECT
  'waste_log_2026' AS source_table,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_dates,
  SUM(CASE WHEN sku IS NULL OR TRIM(sku) = '' THEN 1 ELSE 0 END) AS missing_skus,
  SUM(CASE WHEN units_wasted IS NULL OR units_wasted <= 0 THEN 1 ELSE 0 END) AS invalid_units_wasted
FROM bronze_waste_log_2026
UNION ALL
SELECT
  'daily_consumption_2026', COUNT(*),
  SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN sku IS NULL OR TRIM(sku) = '' THEN 1 ELSE 0 END),
  SUM(CASE WHEN units_used IS NULL OR units_used < 0 THEN 1 ELSE 0 END)
FROM bronze_daily_consumption_2026;

## 4. Verify SKU coverage

Every SKU used in operations should exist in the product master so it can later receive a material cost.

In [ ]:
%sql
SELECT DISTINCT TRIM(c.sku) AS unmatched_sku
FROM bronze_daily_consumption_2026 c
LEFT ANTI JOIN manufacturing_cost_reduction.product_master p
  ON TRIM(c.sku) = TRIM(p.sku)
ORDER BY unmatched_sku;

## Bronze handoff

Proceed to the Silver notebook after raw tables have row counts, expected date ranges, and no unexplained SKU coverage gaps.